# BioME-style student v2: bỏ FiLM, chưng cất hidden + logit + Q/K/V (bản abstract)

**Khác `biome-kd.ipynb`.** Cùng khung student kiểu Llama (GQA, RoPE, SwiGLU, RMSNorm, 12 layer) cho phân loại nhiễu dưới giọng nói, nhưng:

| | `biome-kd.ipynb` | `biome-kd-v2.ipynb` |
|---|---|---|
| FiLM / MSAB | có (vector `h` điều biến mọi layer) | **bỏ** (MSAB chỉ phục vụ FiLM nên bỏ luôn; student là Transformer thuần trên log-mel) |
| Hidden feature | layer 3, 6, 9, 12 (L1 − log σ cos) | giữ nguyên |
| Logit | KL có nhiệt độ + CE | giữ nguyên |
| Q/K/V | không | **layer 3, 6, 9, 12**: chưng cất quan hệ Q·Qᵀ, K·Kᵀ, V·Vᵀ (kiểu MiniLMv2) |
| Import | module trong repo (`scripts`, `src`) | **chỉ `torch`, `torchaudio`**; mọi thành phần định nghĩa ngay trong notebook |

**Luồng dữ liệu**

```
mixture ─► FRCRN (giao diện masking) ─► n̂ = mixture − ŝ
n̂ ─► log-mel 128 ─► patch 16×16 ─► [Layer ×12] ─► RMSNorm ─► mean-pool ─► Linear(36)
layer {3,6,9,12}:  hidden ─► Linear(dim→768)  ~ hidden teacher         (L1 − log σ cos)
layer {3,6,9,12}:  q, k, v (sau Linear, trước RoPE) ~ q, k, v teacher    (KL của ma trận quan hệ)
teacher: n̂ ─► BEATs đã finetune (đóng băng)   (chỉ dùng khi huấn luyện)
```

**Mỗi layer** chỉ còn hai khối con có residual: `RMSNorm → GQA (RoPE)` và `RMSNorm → FFN SiLU có cổng`.

**Trạng thái.** Bản *abstract*: định nghĩa mô-đun, loss, một bước tối ưu trên dữ liệu giả. Chưa nạp dữ liệu, chưa huấn luyện, **chưa chạy** (máy local chưa có `torch`). FRCRN và teacher là *giao diện*; `ToyTeacher` chỉ là bản thế chỗ có đúng shape của BEATs để kiểm tra hook Q/K/V.

In [ ]:
import math
from dataclasses import dataclass
from functools import lru_cache
from typing import NamedTuple

import torch
import torch.nn.functional as F
import torchaudio.functional as taf
from torch import Tensor, nn
from tqdm.auto import tqdm

BIO_VARIANT = "edge"          # "edge" (~6M, dim 192) hoặc "small" (dim 384); xem BIO_VARIANTS


@dataclass(frozen=True)
class BioConfig:
    dim: int = 192              # chiều ẩn: Edge 192, Small 384
    layers: int = 12            # bằng độ sâu của teacher BEATs
    heads: int = 3              # head_dim = 64
    kv_heads: int = 1           # GQA: 3 query head chia sẻ 1 nhóm key/value
    ffn_hidden: int = 512
    n_mels: int = 128
    patch: int = 16
    teacher_dim: int = 768
    kd_layers: tuple = (3, 6, 9, 12)       # layer chưng cất hidden feature
    qkv_layers: tuple = (3, 6, 9, 12)      # layer chưng cất Q/K/V (cùng chỉ số ở student và teacher)
    rel_heads: int = 12                    # số head quan hệ A_r dùng chung cho student và teacher (MiniLMv2)
    num_classes: int = 36
    sample_rate: int = 16000

    def __post_init__(self):
        assert self.dim % self.heads == 0 and self.heads % self.kv_heads == 0
        assert self.dim % self.rel_heads == 0 and self.teacher_dim % self.rel_heads == 0
        assert set(self.kd_layers) | set(self.qkv_layers) <= set(range(1, self.layers + 1))


# Bài BioME nêu Edge 6M/dim 192, Small 26M, chiều sâu 12; số head, nhóm KV, FFN là lựa chọn của người viết.
BIO_VARIANTS = {
    "edge": dict(dim=192, heads=3, kv_heads=1, ffn_hidden=512),
    "small": dict(dim=384, heads=6, kv_heads=2, ffn_hidden=1024),
}

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: Tensor) -> Tensor:
        rms = x.float().square().mean(dim=-1, keepdim=True).add(self.eps).rsqrt()
        return (x.float() * rms).type_as(x) * self.weight


@lru_cache(maxsize=8)
def rope_cache(length: int, head_dim: int, device, base: float = 10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    angles = torch.outer(torch.arange(length, device=device).float(), inv_freq)   # (L, head_dim/2)
    return angles.cos(), angles.sin()


def apply_rope(x: Tensor, cos: Tensor, sin: Tensor) -> Tensor:
    # x: (B, H, L, head_dim); xoay từng cặp chiều (2i, 2i+1) theo góc phụ thuộc vị trí
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack((x1 * cos - x2 * sin, x1 * sin + x2 * cos), dim=-1).flatten(-2)


class GQA(nn.Module):
    def __init__(self, cfg: BioConfig):
        super().__init__()
        self.h, self.g, self.hd = cfg.heads, cfg.kv_heads, cfg.dim // cfg.heads
        self.q = nn.Linear(cfg.dim, self.h * self.hd, bias=False)
        self.k = nn.Linear(cfg.dim, self.g * self.hd, bias=False)
        self.v = nn.Linear(cfg.dim, self.g * self.hd, bias=False)
        self.o = nn.Linear(cfg.dim, cfg.dim, bias=False)

    def _expand(self, t: Tensor) -> Tensor:
        # (B, L, g*hd) -> (B, L, h*hd): lặp mỗi nhóm KV cho các query head của nó, để q, k, v cùng chiều khi so quan hệ
        b, length, _ = t.shape
        return t.view(b, length, self.g, self.hd).repeat_interleave(self.h // self.g, dim=2).reshape(b, length, self.h * self.hd)

    def forward(self, x: Tensor, cos: Tensor, sin: Tensor, return_qkv: bool = False):
        b, length, _ = x.shape
        q, k, v = self.q(x), self.k(x), self.v(x)                    # ★ điểm lấy Q/K/V: ngay sau ba Linear, trước RoPE và softmax
        tap = (q, self._expand(k), self._expand(v)) if return_qkv else None
        q = q.view(b, length, self.h, self.hd).transpose(1, 2)
        k = k.view(b, length, self.g, self.hd).transpose(1, 2)
        v = v.view(b, length, self.g, self.hd).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        try:        # torch >= 2.5: SDPA tự chia sẻ KV theo nhóm
            out = F.scaled_dot_product_attention(q, k, v, enable_gqa=True)
        except TypeError:
            k, v = k.repeat_interleave(self.h // self.g, dim=1), v.repeat_interleave(self.h // self.g, dim=1)
            out = F.scaled_dot_product_attention(q, k, v)
        return self.o(out.transpose(1, 2).reshape(b, length, -1)), tap


class SwiGLU(nn.Module):
    # FFN kiểu Llama: W3( SiLU(W1 x) * (W2 x) )
    def __init__(self, cfg: BioConfig):
        super().__init__()
        self.w1 = nn.Linear(cfg.dim, cfg.ffn_hidden, bias=False)
        self.w2 = nn.Linear(cfg.dim, cfg.ffn_hidden, bias=False)
        self.w3 = nn.Linear(cfg.ffn_hidden, cfg.dim, bias=False)

    def forward(self, x: Tensor) -> Tensor:
        return self.w3(F.silu(self.w1(x)) * self.w2(x))


class BioLayer(nn.Module):
    # v2: không còn FiLM; chỉ attention và FFN, đều có residual.
    def __init__(self, cfg: BioConfig):
        super().__init__()
        self.norm_attn, self.norm_ffn = RMSNorm(cfg.dim), RMSNorm(cfg.dim)
        self.attn, self.ffn = GQA(cfg), SwiGLU(cfg)

    def forward(self, x: Tensor, cos: Tensor, sin: Tensor, return_qkv: bool = False):
        delta, tap = self.attn(self.norm_attn(x), cos, sin, return_qkv)
        x = x + delta
        return x + self.ffn(self.norm_ffn(x)), tap

In [ ]:
class BioFrontend(nn.Module):
    # n̂ (B, N) -> fbank (B, T, n_mels) cho patch embedding. Khả vi. v2 không còn nhánh MSAB (chỉ phục vụ FiLM).
    def __init__(self, cfg: BioConfig, n_fft: int = 400, hop: int = 160, fmin: float = 20.0,
                 mean: float = 15.41663, std: float = 6.55582):
        super().__init__()
        self.n_fft, self.hop, self.mean, self.std = n_fft, hop, mean, std
        self.register_buffer("window", torch.hann_window(n_fft))
        self.register_buffer("fb", taf.melscale_fbanks(n_fft // 2 + 1, fmin, cfg.sample_rate / 2, cfg.n_mels,
                                                       cfg.sample_rate, norm=None, mel_scale="htk"))

    def forward(self, wave: Tensor) -> Tensor:
        with torch.autocast(device_type=wave.device.type, enabled=False):      # STFT n_fft=400 luôn chạy fp32
            x = wave.float() * 32768.0
            x = torch.cat([x[:, :1], x[:, 1:] - 0.97 * x[:, :-1]], dim=1)       # pre-emphasis như fbank của BEATs
            spec = torch.stft(x, self.n_fft, self.hop, window=self.window, center=False, return_complex=True)
            mel = (spec.real.square() + spec.imag.square()).transpose(1, 2) @ self.fb      # (B, T, n_mels)
            return (torch.log(mel.clamp_min(1e-7)) - self.mean) / (2 * self.std)


class MaskedFrontEnd(nn.Module):
    # Giao diện FRCRN dạng masking: `mask_net(real, imag) -> (mask_re, mask_im)` là FRCRN thật khi triển khai.
    def __init__(self, mask_net: nn.Module, n_fft: int = 640, hop: int = 320):
        super().__init__()
        self.mask_net, self.n_fft, self.hop = mask_net, n_fft, hop
        self.register_buffer("window", torch.hann_window(n_fft).sqrt())

    def forward(self, mixture: Tensor):
        spec = torch.stft(mixture, self.n_fft, self.hop, window=self.window, return_complex=True)   # (B, F, T)
        mask_re, mask_im = self.mask_net(spec.real, spec.imag)
        speech = torch.complex(spec.real * mask_re - spec.imag * mask_im, spec.real * mask_im + spec.imag * mask_re)
        s_hat = torch.istft(speech, self.n_fft, self.hop, window=self.window, length=mixture.shape[-1])
        return mixture - s_hat, s_hat                                                         # (n̂, ŝ)


class HalfMask(nn.Module):
    # Mặt nạ giả chỉ để kiểm tra shape và gradient; thay bằng FRCRN thật.
    def __init__(self):
        super().__init__()
        self.scale = nn.Parameter(torch.tensor(0.5))

    def forward(self, real: Tensor, imag: Tensor):
        return self.scale * torch.ones_like(real), torch.zeros_like(imag)

In [ ]:
class StudentOutput(NamedTuple):
    logits: Tensor
    projected: dict      # {k: (B, L, teacher_dim)} cho k trong cfg.kd_layers
    qkv: dict            # {k: (q, k, v)}, mỗi tensor (B, L, dim), cho k trong cfg.qkv_layers (k, v đã lặp lên số query head)
    noise_hat: Tensor


class BioStudent(nn.Module):
    def __init__(self, front: nn.Module, cfg: BioConfig = BioConfig()):
        super().__init__()
        self.front, self.cfg = front, cfg
        self.frontend = BioFrontend(cfg)
        self.patch = nn.Conv2d(1, cfg.dim, cfg.patch, stride=cfg.patch)          # = chiếu tuyến tính từng patch 16×16
        self.layers = nn.ModuleList(BioLayer(cfg) for _ in range(cfg.layers))
        self.norm = RMSNorm(cfg.dim)
        self.head = nn.Linear(cfg.dim, cfg.num_classes)
        self.proj = nn.ModuleDict({str(k): nn.Linear(cfg.dim, cfg.teacher_dim) for k in cfg.kd_layers})
        # Q/K/V không cần lớp chiếu: ma trận quan hệ L×L không phụ thuộc chiều đặc trưng.

    def forward(self, mixture: Tensor, noise_hat: Tensor | None = None) -> StudentOutput:
        if noise_hat is None:                  # truyền n̂ vào để khỏi chạy lại FRCRN khi đã tính ở nơi khác
            noise_hat, _ = self.front(mixture)
        fbank = self.frontend(noise_hat)
        x = self.patch(fbank.unsqueeze(1)).flatten(2).transpose(1, 2)           # (B, Tp*8, dim), token t·8 + f như BEATs
        cos, sin = rope_cache(x.shape[1], self.cfg.dim // self.cfg.heads, x.device)
        projected, qkv = {}, {}
        for i, layer in enumerate(self.layers, start=1):
            x, tap = layer(x, cos, sin, return_qkv=i in self.cfg.qkv_layers)
            if i in self.cfg.kd_layers:
                projected[i] = self.proj[str(i)](x)
            if tap is not None:
                qkv[i] = tap
        return StudentOutput(self.head(self.norm(x).mean(dim=1)), projected, qkv, noise_hat)

## Teacher: giao diện và vị trí lấy Q/K/V

`TeacherOutput` là hợp đồng giữa teacher và loss: logit, hidden theo layer, và Q/K/V theo layer, tất cả cùng lưới token 24×8 = 192 như student.

- **BEATs thật:** đăng ký `register_forward_hook` lên ba Linear `q_proj`, `k_proj`, `v_proj` của `GatedRelPosAttention` ở các layer `cfg.qkv_layers`. Đây là điểm *trước* cổng `gru_rel_pos` và bias vị trí tương đối. Hook lấy đầu ra Linear; BEATs của repo là batch-first nên tensor đã có dạng `B×T×C`, không cần chuyển trục. Hidden lấy từ đầu ra layer như trước.
- **`ToyTeacher` bên dưới** chỉ có đúng shape và vị trí tap của BEATs (12 layer, dim 768, 12 head) với trọng số ngẫu nhiên, để chạy kiểm tra shape/loss mà không cần checkpoint.

In [ ]:
class TeacherOutput(NamedTuple):
    logits: Tensor       # (B, 36)
    hidden: dict         # {k: (B, L, teacher_dim)} với k trong cfg.kd_layers
    qkv: dict            # {k: (q, k, v)}, mỗi tensor (B, L, teacher_dim), k trong cfg.qkv_layers


class TeacherLayer(nn.Module):
    # Post-norm như BEATs; q/k/v lấy ngay sau ba Linear, trước attention.
    def __init__(self, dim: int, heads: int, ffn: int):
        super().__init__()
        self.h = heads
        self.q_proj, self.k_proj, self.v_proj, self.out_proj = (nn.Linear(dim, dim) for _ in range(4))
        self.ln1, self.ln2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.fc1, self.fc2 = nn.Linear(dim, ffn), nn.Linear(ffn, dim)

    def forward(self, x: Tensor):
        b, length, d = x.shape
        q, k, v = self.q_proj(x), self.k_proj(x), self.v_proj(x)                 # ★ tap
        heads = lambda t: t.view(b, length, self.h, d // self.h).transpose(1, 2)
        a = F.scaled_dot_product_attention(heads(q), heads(k), heads(v))
        x = self.ln1(x + self.out_proj(a.transpose(1, 2).reshape(b, length, d)))
        return self.ln2(x + self.fc2(F.gelu(self.fc1(x)))), (q, k, v)


class ToyTeacher(nn.Module):
    # Thế chỗ BEATs đã finetune: n̂ -> TeacherOutput. Đóng băng, chạy dưới no_grad.
    def __init__(self, cfg: BioConfig, heads: int = 12, ffn: int = 1536):
        super().__init__()
        self.cfg = cfg
        self.frontend = BioFrontend(cfg)
        self.patch = nn.Conv2d(1, cfg.teacher_dim, cfg.patch, stride=cfg.patch)
        self.layers = nn.ModuleList(TeacherLayer(cfg.teacher_dim, heads, ffn) for _ in range(cfg.layers))
        self.head = nn.Linear(cfg.teacher_dim, cfg.num_classes)
        self.requires_grad_(False)

    def train(self, mode: bool = True):
        return super().train(False)

    @torch.no_grad()
    def forward(self, noise_hat: Tensor) -> TeacherOutput:
        x = self.patch(self.frontend(noise_hat).unsqueeze(1)).flatten(2).transpose(1, 2)
        hidden, qkv = {}, {}
        for i, layer in enumerate(self.layers, start=1):
            x, tap = layer(x)
            if i in self.cfg.kd_layers:
                hidden[i] = x
            if i in self.cfg.qkv_layers:
                qkv[i] = tap
        return TeacherOutput(self.head(x.mean(dim=1)), hidden, qkv)

In [ ]:
import sys
from pathlib import Path

import yaml

ROOT = Path("/marimo") if (Path("/marimo") / "src").is_dir() else Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.model.beat import BEATClassifier
from src.model.frcrn import FRCRNSeparator
from src.model.reviewkd import rms_normalize
from src.utils.checkpoint import load_checkpoint

FRCRN_E37 = ROOT / "pretrained" / "FRCRN" / "best_e037_20261020T203958Z.pt"                      # FRCRN epoch 37
BEATS_FT = ROOT / "output" / "beats-frcrn-ft" / "checkpoint" / "beat_frcrn" / "best.pt"            # BEATs finetune trên n̂ của FRCRN e37
TARGET_RMS = float(yaml.safe_load((ROOT / "config" / "reviewkd.yaml").read_text(encoding="utf-8"))["review"]["target_rms"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class FRCRNFront(nn.Module):
    # front(mixture) -> (n̂, ŝ). FRCRNSeparator (target=speech) trả thẳng n̂; n̂ được chuẩn hoá RMS như đầu vào của teacher.
    def __init__(self, separator: nn.Module, target_rms: float):
        super().__init__()
        self.separator, self.target_rms = separator, target_rms

    def train(self, mode: bool = True):
        return super().train(False)                      # FRCRN có BatchNorm: bản đóng băng luôn ở eval

    def forward(self, mixture: Tensor):
        noise_hat = self.separator(mixture)
        return rms_normalize(noise_hat, self.target_rms), mixture - noise_hat


class BioTeacher(nn.Module):
    # n̂ -> BEATs đã finetune (đóng băng). Q/K/V lấy bằng forward hook trên q_proj/k_proj/v_proj của self_attn ở các layer cfg.qkv_layers.
    # BEATs của repo là batch-first nên đầu ra Linear đã có dạng (B, T, 768), không cần chuyển trục.
    def __init__(self, beats: BEATClassifier, cfg: BioConfig):
        super().__init__()
        self.beats, self.cfg, self._tap = beats, cfg, {}
        for layer in cfg.qkv_layers:
            attn = beats.backbone.encoder.layers[layer - 1].self_attn
            for name in ("q_proj", "k_proj", "v_proj"):
                getattr(attn, name).register_forward_hook(self._saver(layer, name))

    def _saver(self, layer: int, name: str):
        def hook(_module, _inputs, output):
            self._tap[(layer, name)] = output
        return hook

    def train(self, mode: bool = True):
        return super().train(False)

    @torch.no_grad()
    def forward(self, noise_hat: Tensor) -> TeacherOutput:
        self._tap.clear()
        out = self.beats(noise_hat, layers=self.cfg.kd_layers)
        qkv = {i: tuple(self._tap[(i, n)] for n in ("q_proj", "k_proj", "v_proj")) for i in self.cfg.qkv_layers}
        return TeacherOutput(out.logits, dict(zip(self.cfg.kd_layers, out.hidden)), qkv)


def build_real_models(cfg: BioConfig):
    # Trả (student, teacher). Hai bên dùng chung một FRCRN đóng băng; student.forward gọi front, teacher nhận n̂ đã detach.
    with tqdm(total=3, desc="dựng student/teacher", unit="bước") as bar:
        for path in (FRCRN_E37, BEATS_FT):
            if not path.is_file():
                raise FileNotFoundError(f"thiếu checkpoint: {path}")
        separator = FRCRNSeparator.from_yaml(ROOT / "config" / "frcrn.yaml")
        separator.load_pretrained(FRCRN_E37)
        front = FRCRNFront(separator.to(device).freeze(), TARGET_RMS)
        print(f"  FRCRN e37: {separator.num_parameters() / 1e6:.1f}M tham số, đóng băng"); bar.update()
        beats = BEATClassifier.from_yaml(ROOT / "config" / "beat.yaml")
        load_checkpoint(BEATS_FT, beats)
        beats.to(device).freeze()
        print(f"  BEATs finetune: {beats.num_parameters() / 1e6:.1f}M tham số, đóng băng"); bar.update()
        student = BioStudent(front, cfg).to(device)
        print(f"  student encoder: {sum(p.numel() for n, p in student.named_parameters() if not n.startswith('front.')) / 1e6:.2f}M tham số"); bar.update()
    return student, BioTeacher(beats, cfg)

## Loss

Tổng loss = `w_ce·CE + w_kd·KL(logit) + w_feat·hidden + w_qkv·QKV`.

- **Hidden** (như v1): `mean_k mean_t [ mean_d |ẑ − z| − log σ(cos(ẑ, z)) ]` sau chiếu `dim → 768`.
- **Logit**: KL giữa softmax có nhiệt độ T = 2 của teacher và student, nhân T².
- **Q/K/V (mới, kiểu MiniLMv2).** Với mỗi layer trong `qkv_layers` và mỗi `X ∈ {Q, K, V}`: gom `X` thành `A_r = rel_heads` head quan hệ (student: `dim/A_r` chiều mỗi head, teacher: `768/A_r`), dựng `R = softmax(X_h X_hᵀ / √d_r)` kích thước `L×L` và tối thiểu `KL(R_teacher ‖ R_student)`. Vì chỉ so ma trận quan hệ nên student và teacher khác số head hay khác chiều đều được, và không cần lớp chiếu. Giá trị là trung bình theo batch, head và hàng, rồi theo Q, K, V và theo layer.
- Lấy Q/K/V *trước RoPE* ở student (và trước bias vị trí ở teacher) là lựa chọn của người viết, chưa kiểm chứng bằng thí nghiệm.

In [ ]:
def layerwise_kd_loss(projected: dict, teacher_hidden: dict):
    # L = mean_k mean_t [ mean_d |ẑ - z| - log sigmoid(cos(ẑ, z)) ]   (công thức chưng cất của BioME)
    terms = {}
    for k, z_hat in projected.items():
        z = teacher_hidden[k].detach().float()
        z_hat = z_hat.float()
        l1 = (z_hat - z).abs().mean(dim=-1)
        terms[k] = (l1 - F.logsigmoid(F.cosine_similarity(z_hat, z, dim=-1))).mean()
    return torch.stack(list(terms.values())).mean(), terms


def relation_log_probs(x: Tensor, heads: int) -> Tensor:
    # (B, L, D) -> (B, heads, L, L): log softmax(X_h X_hᵀ / sqrt(d_r)) theo hàng, d_r = D / heads
    b, length, d = x.shape
    x = x.float().view(b, length, heads, d // heads).transpose(1, 2)
    return F.log_softmax(x @ x.transpose(-1, -2) / math.sqrt(d // heads), dim=-1)


def qkv_relation_loss(student_qkv: dict, teacher_qkv: dict, heads: int):
    # KL(R_teacher || R_student) cho Q-Q, K-K, V-V ở từng layer trong student_qkv.
    terms = {}
    for layer, s_tuple in student_qkv.items():
        per_kind = []
        for s, t in zip(s_tuple, teacher_qkv[layer]):
            log_s = relation_log_probs(s, heads)
            log_t = relation_log_probs(t.detach(), heads)
            per_kind.append(F.kl_div(log_s, log_t, reduction="none", log_target=True).sum(dim=-1).mean())
        terms[layer] = torch.stack(per_kind).mean()
    return torch.stack(list(terms.values())).mean(), terms


def total_loss(out: StudentOutput, teacher: TeacherOutput, labels: Tensor, rel_heads: int,
               w_ce: float = 1.0, w_kd: float = 1.0, w_feat: float = 1.0, w_qkv: float = 1.0, temperature: float = 2.0):
    logits = out.logits.float()                  # loss luôn tính fp32 dù encoder chạy bf16
    ce = F.cross_entropy(logits, labels, label_smoothing=0.1)
    kd = F.kl_div(F.log_softmax(logits / temperature, -1), F.softmax(teacher.logits.float() / temperature, -1),
                  reduction="batchmean") * temperature**2
    feat, per_feat = layerwise_kd_loss(out.projected, teacher.hidden)
    rel, per_rel = qkv_relation_loss(out.qkv, teacher.qkv, rel_heads)
    parts = {"ce": ce, "kd": kd, "feat": feat, "qkv": rel,
             **{f"feat_layer{k}": v for k, v in per_feat.items()}, **{f"qkv_layer{k}": v for k, v in per_rel.items()}}
    return w_ce * ce + w_kd * kd + w_feat * feat + w_qkv * rel, parts


def distill_step(student: BioStudent, teacher: nn.Module, optimizer, mixture: Tensor, labels: Tensor, **weights):
    # Một bước huấn luyện. Student chạy FRCRN; teacher nhận đúng n̂ đó (detach) nên FRCRN chỉ chạy một lần.
    student.train()
    out = student(mixture)
    teacher_out = teacher(out.noise_hat.detach())
    loss, parts = total_loss(out, teacher_out, labels, student.cfg.rel_heads, **weights)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_([p for p in student.parameters() if p.requires_grad], 1.0)
    optimizer.step()
    return loss, parts


def train_epoch(student: BioStudent, teacher: nn.Module, optimizer, loader, epoch: int, epochs: int, **weights) -> dict:
    # Một epoch chưng cất. Thanh tqdm theo batch hiển thị loss và bốn thành phần; cuối epoch in trung bình.
    device = next(student.parameters()).device
    names = ("loss", "ce", "kd", "feat", "qkv")
    sums, steps = dict.fromkeys(names, 0.0), 0
    bar = tqdm(loader, desc=f"epoch {epoch}/{epochs} | train", unit="batch", leave=False, dynamic_ncols=True)
    for mixture, labels in bar:
        loss, parts = distill_step(student, teacher, optimizer, mixture.to(device), labels.to(device), **weights)
        steps += 1
        values = {"loss": float(loss), **{k: float(parts[k]) for k in names[1:]}}
        for k, v in values.items():
            sums[k] += v
        bar.set_postfix({k: f"{v:.3f}" for k, v in values.items()})
    means = {k: v / max(steps, 1) for k, v in sums.items()}
    tqdm.write(f"epoch {epoch}/{epochs} | " + " ".join(f"{k} {v:.4f}" for k, v in means.items()))
    return means


In [ ]:
# Kiểm tra shape, gradient và một bước tối ưu trên dữ liệu giả (chưa chạy trong phiên viết notebook này).
_bar = tqdm(total=6, desc="kiểm tra v2", unit="bước")
_cfg = BioConfig(**BIO_VARIANTS[BIO_VARIANT])
print(f"BioME v2 {BIO_VARIANT}: dim {_cfg.dim}, {_cfg.layers} layer, {_cfg.heads} head / {_cfg.kv_heads} KV, FFN {_cfg.ffn_hidden}")
_student = BioStudent(MaskedFrontEnd(HalfMask()), _cfg)
_teacher = ToyTeacher(_cfg)
_bar.set_postfix_str("dựng student/teacher"); _bar.update(1)
_mixture = torch.randn(2, 64000) * 0.1                                  # 4 giây, 16 kHz
_labels = torch.tensor([0, 1])
_out = _student(_mixture)
_bar.set_postfix_str("forward student"); _bar.update(1)
print(f"  logits {tuple(_out.logits.shape)} | hidden layer {sorted(_out.projected)} | qkv layer {sorted(_out.qkv)}")

_frames = (64000 - 400) // 160 + 1                                      # 398 khung
_tokens = (_frames // _cfg.patch) * (_cfg.n_mels // _cfg.patch)             # 24 × 8 = 192, trùng lưới token của BEATs
assert not any("film" in name for name, _ in _student.named_parameters())     # v2 không còn FiLM
assert _out.logits.shape == (2, _cfg.num_classes)
assert all(_out.projected[k].shape == (2, _tokens, _cfg.teacher_dim) for k in _cfg.kd_layers)
assert set(_out.qkv) == set(_cfg.qkv_layers) and all(t.shape == (2, _tokens, _cfg.dim) for tap in _out.qkv.values() for t in tap)

_t_out = _teacher(_out.noise_hat.detach())
_bar.set_postfix_str("forward teacher"); _bar.update(1)
assert set(_t_out.hidden) == set(_cfg.kd_layers) and set(_t_out.qkv) == set(_cfg.qkv_layers)
assert all(t.shape == (2, _tokens, _cfg.teacher_dim) for tap in _t_out.qkv.values() for t in tap)
assert relation_log_probs(_out.qkv[3][0], _cfg.rel_heads).shape == (2, _cfg.rel_heads, _tokens, _tokens)

_loss, _parts = total_loss(_out, _t_out, _labels, _cfg.rel_heads)
_loss.backward()
_bar.set_postfix_str("loss + backward"); _bar.update(1)
assert _student.front.mask_net.scale.grad is not None                    # gradient chảy ngược về FRCRN
print({k: round(float(v), 3) for k, v in _parts.items() if not k.endswith(tuple("0123456789"))})
print("  từng layer |", " ".join(f"{k} {float(v):.3f}" for k, v in _parts.items() if k.endswith(tuple("0123456789"))))

_student.zero_grad(set_to_none=True)
_optimizer = torch.optim.AdamW([p for p in _student.parameters() if p.requires_grad], lr=3e-4, betas=(0.9, 0.98), weight_decay=0.05)
_loss2, _ = distill_step(_student, _teacher, _optimizer, _mixture, _labels)
print("một bước distill_step ok, loss", round(float(_loss2), 3))
_bar.set_postfix_str("distill_step"); _bar.update(1)
_loader = [(_mixture, _labels)] * 3                                      # 3 batch giả để thử train_epoch
_epoch_means = train_epoch(_student, _teacher, _optimizer, _loader, epoch=1, epochs=1)
_bar.set_postfix_str("train_epoch"); _bar.update(1); _bar.close()
print("tham số encoder (không gồm FRCRN):", round(sum(p.numel() for n, p in _student.named_parameters() if not n.startswith("front.")) / 1e6, 2), "M")

In [ ]:
LOAD_REAL = True            # True: dựng FRCRN + BEATs thật và kiểm tra trên CHECK_BATCHES batch validation (cần GPU, checkpoint, dataset)
CHECK_BATCHES = 4

if LOAD_REAL:
    from scripts.finetune import CLIP_SECONDS
    from scripts.load_dataset import Mixture36Dataset, build_dataloader, subsample

    _cfg = BioConfig(**BIO_VARIANTS[BIO_VARIANT])
    REAL_STUDENT, REAL_TEACHER = build_real_models(_cfg)
    _loader = build_dataloader(subsample(Mixture36Dataset(split="validation", duration=CLIP_SECONDS), CHECK_BATCHES * 8), 8, False, 0)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    _sums, _correct, _seen = {}, 0, 0
    _bar = tqdm(_loader, desc="kiểm tra student/teacher", unit="batch", total=CHECK_BATCHES)
    for _batch in _bar:
        _mixture, _labels = _batch["waveform"].to(device), _batch["target"].argmax(dim=1).to(device)
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=device.type == "cuda"):
            _s_out = REAL_STUDENT(_mixture)
        _t_out = REAL_TEACHER(_s_out.noise_hat.detach())
        _loss, _parts = total_loss(_s_out, _t_out, _labels, _cfg.rel_heads)
        _loss.backward()
        _correct += int((_t_out.logits.argmax(-1) == _labels).sum()); _seen += len(_labels)
        for _k in ("ce", "kd", "feat", "qkv"):
            _sums[_k] = _sums.get(_k, 0.0) + float(_parts[_k])
        _bar.set_postfix({k: f"{v / (_bar.n + 1):.3f}" for k, v in _sums.items()})
    print(f"teacher top-1 trên {_seen} clip validation: {_correct / _seen:.4f}")
    print("trung bình loss:", {k: round(v / CHECK_BATCHES, 3) for k, v in _sums.items()})
    if device.type == "cuda":
        print(f"VRAM đỉnh: {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB")
else:
    print("LOAD_REAL = False: bỏ qua kiểm tra student/teacher thật (đặt True trên GPU để chạy).")

## Ghi chú và các điểm chưa chốt

Những chỗ dưới đây là **suy luận của người viết**, không có trong bài BioME:

- **Bỏ FiLM kéo theo bỏ MSAB.** Trong v1, MSAB chỉ đưa thông tin vào qua FiLM. Nếu vẫn muốn thử nhánh phổ modulation, phải thêm nó vào chỗ khác (ví dụ cộng vào token đầu vào).
- RoPE 1 chiều trên chuỗi token đã duỗi theo thời gian (token thứ `t·8 + f`), giữ nguyên như v1.
- **Q/K/V:** `rel_heads = 12` và việc lặp k, v của student lên số query head trước khi so quan hệ là lựa chọn của người viết; điểm lấy (trước RoPE / trước bias vị trí) chưa được so với các lựa chọn khác. Ma trận quan hệ tốn bộ nhớ: `B · A_r · L²` phần tử cho mỗi cặp (layer, Q/K/V); với B = 32, A_r = 12, L = 192, bốn layer, ba loại, student cộng teacher khoảng 1,4 GB ở fp32.
- Trọng số `w_ce`, `w_kd`, `w_feat`, `w_qkv` đặt bằng 1, chưa ablation. Nên tăng dần `w_qkv` thay vì bật hết từ đầu để biết đóng góp riêng.
- Nếu FRCRN của student huấn luyện được, n̂ truyền cho teacher trong `distill_step` là n̂ của student (đã detach), không phải n̂ của bản FRCRN đóng băng. Muốn đúng như v1, cần chạy hai bản FRCRN riêng.
- Số tham số Edge/Small là ước lượng; ô kiểm tra shape in số thật khi chạy.

Chưa làm (cố ý, vì notebook này không import mã ngoài): nạp dữ liệu Mixture-36l, nạp BEATs/FRCRN thật, vòng huấn luyện nhiều epoch, đánh giá 9 chỉ số theo SNR. Có thể thêm bằng cách dựng `DataLoader` rồi gọi `distill_step`.